In [119]:
import pyspark
import csv
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('BigD_OurD').getOrCreate()
sc = spark.sparkContext

In [123]:
########################
# Act 3: Partitioning  #
########################

# Initialization of the whole RDD
game_rdd = sc.textFile("vgsales.csv").map(lambda line: next(csv.reader([line], skipinitialspace=True)))
header = game_rdd.first()
game_rdd = game_rdd.filter(lambda x: x != header)

In [121]:
###################################
# Genre to Global Sale Comparison #
#    by Keiron Mirandilla         #
###################################

# Keep genre(5th) and global sales(last) as a tuple w/ Hash Partitioning
# Get sum of global sales and sort by highest
genre_sales = game_rdd.map(lambda x: (x[4], float(x[-1]))).partitionBy(4).persist()
summed_sales = genre_sales.reduceByKey(lambda a,b: a+b)

genres = summed_sales.collect()

In [122]:
print("Global Sales of Each Genre:")
for i in range(len(genres)):
    print(f"  {i+1}. {genres[i][0]} with {genres[i][1]:.2f} million sales.")

Global Sales of Each Genre:
  1. Role-Playing with 927.37 million sales.
  2. Shooter with 1037.37 million sales.
  3. Fighting with 448.91 million sales.
  4. Platform with 831.37 million sales.
  5. Puzzle with 244.95 million sales.
  6. Strategy with 175.12 million sales.
  7. Sports with 1330.93 million sales.
  8. Racing with 732.04 million sales.
  9. Action with 1751.18 million sales.
  10. Adventure with 239.04 million sales.
  11. Misc with 809.96 million sales.
  12. Simulation with 392.20 million sales.


In [118]:
# Nash A. Mapula

# Pipeline: Filter for 'Action' and clean the Year/Sales data
# use a try/except or a simple check to skip 'N/A' years
def clean_and_map(x):
    try:
        # Year is index 3, Global_Sales is index 10 (or -1)
        return (int(x[3]), float(x[10]))
    except:
        return None

# The Pipeline: Filter Action > Map to (Year, Sales) > Remove None values
yearly_action_sales = data_rdd.filter(lambda x: x[4] == "Action") \
                              .map(clean_and_map) \
                              .filter(lambda x: x is not None)

# Strategy: Range Partitioning
yas_partitioned = yearly_action_sales.sortByKey(numPartitions=8).persist()

# View the Result (Transformation Output)
# see the total sales for the first 5 years in our partitioned data
results = yas_partitioned.reduceByKey(lambda a, b: a + b).sortByKey()
results.take(5)

[(1980, 0.34), (1981, 14.84), (1982, 6.52), (1983, 2.86), (1984, 1.85)]

In [110]:
##############################
# Act 4: Data Preprocessing  #
#        and DataFrames      #
##############################

# Initialization and pre-processing of dataframe and schema
headers = [
    ("Show ID", StringType(), False),
    ("Type", StringType(), False),
    ("Title", StringType(), False),
    ("Director", StringType(), True),
    ("Cast", StringType(), True),
    ("Country", StringType(), True),
    ("Date Added", StringType(), True),
    ("Release Year", IntegerType(), False),
    ("Rating", StringType(), True),
    ("Duration", StringType(), True),
    ("Listed In", StringType(), False),
    ("Description", StringType(), False)
]

netflix_schema = StructType([
    StructField(*field) for field in headers
])
netflix_df = spark.read.csv("netflix_titles.csv", header=True, schema=netflix_schema)

# Get headers with missing values and
kv_nulls = {key[0]: "N/A" for key in headers if key[2]}
netflix_df = netflix_df.fillna(kv_nulls)

In [ ]:
netflix_df.select("*").show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|Show ID|   Type|               Title|            Director|                Cast|             Country|        Date Added|Release Year|Rating| Duration|           Listed In|         Description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                 N/A|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                 N/A|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan